In [52]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse/") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [2]:
data = [("Spring", 12.3),
("Summer", 10.5),
("Autumn", 8.2),
("Winter", 15.1)]

In [3]:
spark.createDataFrame(data, ["season", "windspaeed"])

season,windspaeed
Spring,12.3
Summer,10.5
Autumn,8.2
Winter,15.1


In [4]:
! head ../../datasets/library_data.json

{"library_name": "Central Library","location": "City Center","books": [{"book_id": "B001","book_name": "The Great Gatsby","author": "F. Scott Fitzgerald","copies_available": 5},{"book_id": "B002","book_name": "To Kill a Mockingbird","author": "Harper Lee","copies_available": 3}],"members": [{"member_id": "M001","member_name": "John Smith","age": 28,"books_borrowed": ["B001"]},{"member_id": "M002","member_name": "Emma Johnson","age": 35,"books_borrowed": []}]},
{"library_name": "Community Library","location": "Suburb","books": [{"book_id": "B003","book_name": "1984","author": "George Orwell","copies_available": 2},{"book_id": "B004","book_name": "Pride and Prejudice","author": "Jane Austen","copies_available": 4}],"members": [{"member_id": "M003","member_name": "Michael Brown","age": 42,"books_borrowed": ["B003","B004"]},{"member_id": "M004","member_name": "Sophia Davis","age": 31,"books_borrowed": ["B004"]}]}


In [5]:
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, IntegerType

library_schema = StructType([
    StructField("library_name", StringType()),
    StructField("location", StringType()),
    StructField("books", ArrayType(
        StructType([
            StructField("book_id", StringType()),
            StructField("book_name", StringType()),
            StructField("author", StringType()),
            StructField("copies_available", IntegerType())
        ])
    )),
    StructField("members", ArrayType(
    StructType([
        StructField("member_id", StringType()),
        StructField("member_name", StringType()),
        StructField("age", IntegerType()),
        StructField("books_borrowed", ArrayType(StringType()))
    ])
    ))
])


library_df = spark.read.json("./datasets/library_data.json", library_schema)

In [6]:
library_df.show(truncate=False)

+-----------------+-----------+------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------+
|library_name     |location   |books                                                                                           |members                                                                    |
+-----------------+-----------+------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------+
|Central Library  |City Center|[{B001, The Great Gatsby, F. Scott Fitzgerald, 5}, {B002, To Kill a Mockingbird, Harper Lee, 3}]|[{M001, John Smith, 28, [B001]}, {M002, Emma Johnson, 35, []}]             |
|Community Library|Suburb     |[{B003, 1984, George Orwell, 2}, {B004, Pride and Prejudice, Jane Austen, 4}]                   |[{M003, Michael Brown, 42, [B003, B004]}, {M004, Sop

In [7]:
library_df.printSchema()

root
 |-- library_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- books: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- book_id: string (nullable = true)
 |    |    |-- book_name: string (nullable = true)
 |    |    |-- author: string (nullable = true)
 |    |    |-- copies_available: integer (nullable = true)
 |-- members: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- member_id: string (nullable = true)
 |    |    |-- member_name: string (nullable = true)
 |    |    |-- age: integer (nullable = true)
 |    |    |-- books_borrowed: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)



In [8]:
train_df = spark.read.option("header", True).csv("datasets/train.csv")

In [9]:
display(train_df)

train_number,train_name,seats_available,passenger_name,age,ticket_number,seat_number
123,Express,100,John,25,T123,A1
123,Express,100,Emma,30,T124,B2
456,Superfast,150,Michael,35,T125,C3
456,Superfast,150,Sophia,40,T126,D4
789,Local,50,William,28,T127,E5
789,Local,50,Sophia,32,T128,F6
789,Local,50,Oliver,45,T129,G7


In [10]:
train_df.printSchema()

root
 |-- train_number: string (nullable = true)
 |-- train_name: string (nullable = true)
 |-- seats_available: string (nullable = true)
 |-- passenger_name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- ticket_number: string (nullable = true)
 |-- seat_number: string (nullable = true)



In [11]:
train_drop_df = train_df.drop("passenger_name", "age")

In [12]:
dedup_train_df = train_drop_df.dropDuplicates(["train_number", "ticket_number"])

In [13]:
display(dedup_train_df)

train_number,train_name,seats_available,ticket_number,seat_number
789,Local,50,T128,F6
456,Superfast,150,T125,C3
789,Local,50,T129,G7
123,Express,100,T124,B2
456,Superfast,150,T126,D4
789,Local,50,T127,E5
123,Express,100,T123,A1


In [14]:
dedup_train_df.count()

7

In [15]:
train_df.dropDuplicates(["train_name"]).count()

3

In [16]:
sales_schema = "store_id integer, product string, quantity integer, revenue double"
sales_perm_df = spark.read.format("json") \
.option("mode", "permissive") \
.schema(sales_schema) \
.load("datasets/sales_data.json")

In [17]:
sales_perm_df.count()

22

In [18]:
sales_malF_df = spark.read.format("json") \
.option("mode", "dropmalformed") \
.schema(sales_schema) \
.load("datasets/sales_data.json")

In [19]:
sales_malF_df.count()

21

In [20]:
sales_failF_df = spark.read.format("json") \
.option("mode", "failfast") \
.schema(sales_schema) \
.load("datasets/sales_data.json")

In [21]:
sales_failF_df.show()

Py4JJavaError: An error occurred while calling o178.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 38.0 failed 4 times, most recent failure: Lost task 0.3 in stage 38.0 (TID 821) (w01.itversity.com executor 2): org.apache.spark.SparkException: Malformed records are detected in record parsing. Parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.
	at org.apache.spark.sql.catalyst.util.FailureSafeParser.parse(FailureSafeParser.scala:70)
	at org.apache.spark.sql.execution.datasources.json.TextInputJsonDataSource$.$anonfun$readFile$9(JsonDataSource.scala:144)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:484)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:490)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:458)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:93)
	at scala.collection.Iterator$

In [22]:
! hadoop fs -head datasets/hospital.csv

patient_id,admission_date,discharge_date,diagnosis,doctor_id,total_cost
1,01-01-2022,2022-01-10,Pneumonia,101,5000.00
2,02-05-2022,2022-02-09,Appendicitis,102,7000.00
3,03-12-2022,2022-03-18,Fractured Arm,103,3500.00
4,04-02-2022,2022-04-08,Heart Attack,104,15000.00
5,05-05-2022,2022-05-07,Influenza,105,2500.00
6,06-10-2022,2022-06-15,Appendicitis,106,8000.00
7,07-20-2022,2022-07-25,Pneumonia,107,5500.00
8,08-25-2022,2022-09-01,Heart Attack,108,20000.00
9,09-15-2022,2022-09-22,Fractured Leg,109,6000.00
10,10-05-2022,2022-10-10,Appendicitis,110,7500.00
11,11-02-2022,2022-11-05,Influenza,111,2800.00
12,12-10-2022,2022-12-18,Pneumonia,112,6000.00
13,01-02-2023,2023-01-09,Heart Attack,113,18000.00
14,02-14-2023,2023-02-18,Appendicitis,114,7200.00
15,03-20-2023,2023-03-28,Fractured Arm,115,3800.00
16,04-05-2023,2023-04-11,Influenza,116,2700.00
17,05-08-2023,2023-05-11,Heart Attack,117,16000.00
18,06-15-2023,2023-06-20,Pneumonia,118,4800.00
19,07-22-2023,2023-07-27,Fractured Leg,119,6500.00


In [29]:
from pyspark.sql.types import FloatType, DateType

hospital_schema = StructType([
    StructField("patient_id", IntegerType()),
    StructField("admission_date", DateType()),
    StructField("discharge_date", DateType()),
    StructField("diagnosis", StringType()),
    StructField("doctor_id", IntegerType()),
    StructField("total_cost", FloatType())
])


hospital_df = spark.read.option("header", "true").schema(hospital_schema).csv("datasets/hospital.csv")

In [24]:
hospital_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- admission_date: date (nullable = true)
 |-- discharge_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- doctor_id: integer (nullable = true)
 |-- total_cost: float (nullable = true)



In [30]:
from pyspark.sql.functions import date_format, datediff

cleaned_hospital_df = hospital_df.withColumn("admission_date", date_format("admission_date", "yyyy-MM-dd"))

In [31]:
from pyspark.sql.functions import expr

update_hospital_df = hospital_df.drop("doctor_id")

In [32]:
update_hospital_df.show(10)

Py4JJavaError: An error occurred while calling o225.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 40.0 failed 4 times, most recent failure: Lost task 0.3 in stage 40.0 (TID 829) (w02.itversity.com executor 1): org.apache.spark.SparkUpgradeException: You may get a different result due to the upgrading of Spark 3.0: Fail to parse '01-01-2022' in the new parser. You can set spark.sql.legacy.timeParserPolicy to LEGACY to restore the behavior before Spark 3.0, or set to CORRECTED and treat it as an invalid datetime string.
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:150)
	at org.apache.spark.sql.catalyst.util.DateTimeFormatterHelper$$anonfun$checkParsedDiff$1.applyOrElse(DateTimeFormatterHelper.scala:141)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:38)
	at org.apache.spark.sql.catalyst.util.Iso8601DateFormatter.$anonfun$

### Solution:
Change your schema: Temporarily change admission_date and discharge_date to StringType() in your reader schema.Apply coalesce and to_date: Use coalesce() to try parsing the column with the first format, and if that fails, try the second format.

In [58]:
from pyspark.sql.types import FloatType, DateType

hospital_schema = StructType([
    StructField("patient_id", IntegerType()),
    StructField("admission_date", DateType()),
    StructField("discharge_date", DateType()),
    StructField("diagnosis", StringType()),
    StructField("doctor_id", IntegerType()),
    StructField("total_cost", FloatType())
])


hospital_df = spark.read.option("header", "true").option("dateFormat","MM-dd-yyyy").schema(hospital_schema).csv("datasets/hospital.csv")

In [59]:
hospital_df.show()

+----------+--------------+--------------+-------------+---------+----------+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|
+----------+--------------+--------------+-------------+---------+----------+
|         1|    2022-01-01|    2022-01-10|    Pneumonia|      101|    5000.0|
|         2|    2022-02-05|    2022-02-09| Appendicitis|      102|    7000.0|
|         3|    2022-03-12|    2022-03-18|Fractured Arm|      103|    3500.0|
|         4|    2022-04-02|    2022-04-08| Heart Attack|      104|   15000.0|
|         5|    2022-05-05|    2022-05-07|    Influenza|      105|    2500.0|
|         6|    2022-06-10|    2022-06-15| Appendicitis|      106|    8000.0|
|         7|    2022-07-20|    2022-07-25|    Pneumonia|      107|    5500.0|
|         8|    2022-08-25|    2022-09-01| Heart Attack|      108|   20000.0|
|         9|    2022-09-15|    2022-09-22|Fractured Leg|      109|    6000.0|
|        10|    2022-10-05|    2022-10-10| Appendicitis|      11

In [ ]:
hospital_df.drop("doctor_id")

In [54]:
from pyspark.sql.functions import to_date, date_format

df_clean = hospital_df.withColumn("admission_date", date_format("admission_date", "yyyy-MM-dd"))

In [55]:
df_clean.show()

+----------+--------------+--------------+-------------+---------+----------+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|
+----------+--------------+--------------+-------------+---------+----------+
|         1|          null|    2022-01-10|    Pneumonia|      101|    5000.0|
|         2|          null|    2022-02-09| Appendicitis|      102|    7000.0|
|         3|          null|    2022-03-18|Fractured Arm|      103|    3500.0|
|         4|          null|    2022-04-08| Heart Attack|      104|   15000.0|
|         5|          null|    2022-05-07|    Influenza|      105|    2500.0|
|         6|          null|    2022-06-15| Appendicitis|      106|    8000.0|
|         7|          null|    2022-07-25|    Pneumonia|      107|    5500.0|
|         8|          null|    2022-09-01| Heart Attack|      108|   20000.0|
|         9|          null|    2022-09-22|Fractured Leg|      109|    6000.0|
|        10|          null|    2022-10-10| Appendicitis|      11

In [51]:
spark.stop()